# GéoMarketing IDF : J10b — Commerces alimentaires en Île-de-France

## Objectifs
- Extraire les commerces alimentaires et certaines grandes surfaces de la BPE.
- Conserver les observations détaillées pour la cartographie.
- Compter les équipements par commune et par catégorie.
- Enrichir le profil communal J10.
- Documenter les contrôles et les limites.

## Périmètre
- Groupe alimentaire B2 sélectionné.
- Grandes surfaces B104 et B105 conservées séparément.
- Restaurants déjà traités dans J6/J9b.

## Limites
- Les comptages portent sur des observations BPE.
- La présence dans la BPE ne confirme pas une ouverture actuelle.
- Le nombre de commerces ne mesure ni la fréquentation ni le chiffre d'affaires.
- Une offre de restauration à emporter doit être vérifiée sur le terrain.

In [1]:
# Import des librairies

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Pandas :", pd.__version__)

Pandas : 2.2.2


In [2]:
#Définir les dossiers

RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_INTERIM = RACINE / "data" / "interim" / "j10b"
DOSSIER_PROCESSED = RACINE / "data" / "processed"

FICHIER_BPE = RACINE / "data" / "raw" / "bpe" / "BPE25.csv"

FICHIER_PROFIL = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10.csv"
)

FICHIER_DETAIL = (
    DOSSIER_PROCESSED / "commerces_alimentaires_idf_j10b.csv"
)

FICHIER_COMMUNES = (
    DOSSIER_INTERIM / "commerces_alimentaires_communes.csv"
)

FICHIER_SORTIE = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10b.csv"
)

for fichier in [FICHIER_BPE, FICHIER_PROFIL]:
    if not fichier.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {fichier}")

DOSSIER_INTERIM.mkdir(parents=True, exist_ok=True)
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

print("BPE :", FICHIER_BPE)
print("Profil d'entrée :", FICHIER_PROFIL)
print("Profil de sortie :", FICHIER_SORTIE)

BPE : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\bpe\BPE25.csv
Profil d'entrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10.csv
Profil de sortie : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10b.csv


In [3]:
#Fonctions utiles

def normaliser_nom_colonne(nom):
    nom = unicodedata.normalize(
        "NFKD", str(nom).strip().upper()
    )
    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^A-Z0-9]+", "_", nom).strip("_")


def nettoyer_texte(serie):
    resultat = serie.astype("string").str.strip()

    valeurs_manquantes = {
        "", "NA", "NAN", "NONE", "NULL",
        "<NA>", "_U", "_Z", "[ND]",
    }

    return resultat.mask(
        resultat.str.upper().isin(valeurs_manquantes)
    )


def normaliser_code_commune(serie):
    resultat = nettoyer_texte(serie).str.upper()
    resultat = resultat.str.replace(r"\.0$", "", regex=True)

    est_numerique = resultat.str.fullmatch(
        r"\d{1,5}", na=False
    )

    return resultat.where(
        ~est_numerique,
        resultat.str.zfill(5),
    )


def convertir_nombre(serie):
    return pd.to_numeric(
        nettoyer_texte(serie)
        .str.replace("\u202f", "", regex=False)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def sauvegarder_csv(table, chemin):
    table.to_csv(
        chemin,
        index=False,
        encoding="utf-8-sig",
    )


def sha256_fichier(chemin):
    empreinte = hashlib.sha256()

    with open(chemin, "rb") as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b""):
            empreinte.update(bloc)

    return empreinte.hexdigest()

In [4]:
#Charger le profil J10

profil = pd.read_csv(
    FICHIER_PROFIL,
    dtype="string",
    encoding="utf-8-sig",
)

profil.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil.columns
]

if profil.columns.duplicated().any():
    raise ValueError("Noms de colonnes dupliqués dans le profil.")

if "CODGEO" not in profil.columns:
    raise ValueError("La colonne CODGEO manque dans le profil J10.")

profil["CODGEO"] = normaliser_code_commune(profil["CODGEO"])

codes_valides = profil["CODGEO"].str.fullmatch(
    r"\d{5}", na=False
)

if not codes_valides.all():
    raise ValueError("Codes communaux manquants ou invalides.")

if profil["CODGEO"].duplicated().any():
    raise ValueError("Plusieurs lignes existent pour une même commune.")

DEPARTEMENTS_IDF = {
    "75", "77", "78", "91", "92", "93", "94", "95"
}

if not profil["CODGEO"].str[:2].isin(DEPARTEMENTS_IDF).all():
    raise ValueError("Le profil contient des codes hors Île-de-France.")

codes_profil = set(profil["CODGEO"])

print("Nombre de communes :", len(profil))
display(profil.head())

Nombre de communes : 1266


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,...,NB_LIEUX_BUS,NB_STATIONS_METRO,NB_STATIONS_TRAMWAY,NB_GARES_RER,NB_GARES_TRAIN,NB_STATIONS_FUNICULAIRE,NB_LIEUX_FERRES,NB_GARES_STATIONS_TRANSPORT_LOURD,NB_LIEUX_MULTIMODAUX,NB_POLES_MULTIMODAUX_LOURDS,NB_LIGNES_TRANSPORT,NB_MODES_TRANSPORT_PRESENTS,NB_OPERATEURS_TRANSPORT,NB_LIGNES_BUS,NB_LIGNES_METRO,NB_LIGNES_TRAMWAY,NB_LIGNES_RER,NB_LIGNES_TRAIN,NB_LIGNES_FUNICULAIRE,NB_LIGNES_AUTRES,NB_LIGNES_FERREES,NB_LIGNES_TRANSPORT_LOURD,A_BUS,A_METRO,A_TRAMWAY,A_RER,A_TRAIN,A_TRANSPORT_LOURD,NB_LIEUX_TRANSPORT_10000_HAB,NB_LIGNES_TRANSPORT_10000_HAB,NB_LIGNES_FERREES_10000_HAB,NB_GARES_STATIONS_LOURDES_10000_HAB,NB_POLES_MULTIMODAUX_10000_HAB,INDICE_LIEUX_TRANSPORT_IDF_BASE100,INDICE_LIGNES_FERREES_IDF_BASE100,NB_LIGNES_TRANSPORT_1000_EMPLOIS,NB_GARES_STATIONS_LOURDES_1000_EMPLOIS,CLASSE_DESSERTE_TRANSPORT,NIVEAU_VIE_MEDIAN_2023,TAUX_PAUVRETE_60_2023,STATUT_NIVEAU_VIE_2023,STATUT_PAUVRETE_2023,MILLESIME_REVENUS,GEOGRAPHIE_REVENUS,SOURCE_REVENUS,REVENU_SOURCE_APPARIEE_2023,INDICE_NIVEAU_VIE_MEDIAN_IDF_BASE100_2023,ECART_TAUX_PAUVRETE_IDF_POINTS_2023,NIVEAU_VIE_DISPONIBLE_2023,PAUVRETE_DISPONIBLE_2023
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,787667.317537046,970898.414592524,481426.8085490755,178607.31443995849,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.5,-0.59,33650.0,16.8,1551133.75022,188002.0,73.38459010221389,88.94429449710343,20911,8329,12553,29,0,98.93055085738077,39.40474191053151,59.38860910108081,39.8307111089857,60.03060590120033,200.61215141394868,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,...,1181,249,60,12,11,2,305,261,208,182,251,7,33,199,19,4,6,17,1,5,47,42,1,1,1,1,1,1,6.14087585542921,1.1874883202717503,0.2223583707281763,1.2347986119160432,0.8610473079261297,40.39419981645854,50.88744885553059,0.1386027134252865,0.1441247338804771,POLE_MULTIMODAL_LOURD,33650.0,16.8,DISPONIBLE,DISPONIBLE,2023,2026-01-01,INSEE_FILOSOFI_2,1,119.28394186458702,-0.5,1,1
1,77001,Achères-la-Forêt,77,11,1232.0,1139.0,1183.0,182.135094788161,159.522993092748,189.869837113436,347.331906368833,204.113784168138,89.442063221765,10.5843212469189,341.658087880909,349.39283020618404,304.1401686368219,100.0263844686839,15.4,13.48,16.05,28.88,29.53,25.71,8.46,44.0,3.86,0.63,34840.0,<NA>,130.30679,35.0,11.014944209636518,29.585798816568047,1,0,1,0,0,8.4530853761623,0.0,8.4530853761623,0.0,100.0,0.0,32.18226,25.01503,65.89736,58.31802,48.54337,...,4,0,0,0,0,0,0,0,0,0,2,1,1,2,0,0,0,0,0,0,0,0,1,0,0,0,0,0,33.8123415046492,16.9061707523246,0.0,0.0,0.0,222.4149308919177,0.0,10.382951816095082,0.0,BUS_UNIQUEMENT,34840.0,<NA>,DISPONIBLE,CONFIDENTIELLE,2023,2026-01-01,INSEE_FILOSOFI_2,1,123.50230414746544,<NA>,1,0
2,77002,Amillis,77,11,781.0,819.0,821.0,127.936213516012,122.351987081097,146.913593456989,151.60336129515,188.399316742963,59.3206016148823,24.4749262929075,250.288200597

In [5]:
#Définir les catégories

CATEGORIES = {
    "B104": "HYPER_GRANDS_MAGASINS",
    "B105": "SUPER_MULTI_COMMERCES",
    "B201": "SUPERETTES",
    "B202": "EPICERIES",
    "B204": "BOUCHERIES",
    "B205": "SURGELES",
    "B206": "POISSONNERIES",
    "B207": "BOULANGERIES_PATISSERIES",
    "B208": "FRUITS_LEGUMES",
    "B209": "BOISSONS",
    "B210": "AUTRES_ALIMENTAIRES",
}

CODES_ALIMENTAIRES_B2 = [
    code for code in CATEGORIES if code.startswith("B2")
]

CODES_GRANDES_SURFACES = ["B104", "B105"]

COLONNES_COMPTAGE = {
    code: f"NB_OBS_{categorie}_BPE2025"
    for code, categorie in CATEGORIES.items()
}

display(
    pd.DataFrame(
        CATEGORIES.items(),
        columns=["TYPEQU", "CATEGORIE_ANALYSE"],
    )
)

,TYPEQU,CATEGORIE_ANALYSE
0,B104,HYPER_GRANDS_MAGASINS
1,B105,SUPER_MULTI_COMMERCES
2,B201,SUPERETTES
3,B202,EPICERIES
4,B204,BOUCHERIES
5,B205,SURGELES
6,B206,POISSONNERIES
7,B207,BOULANGERIES_PATISSERIES
8,B208,FRUITS_LEGUMES
9,B209,BOISSONS


In [6]:
#Vérifier le fichier avant de charger

with open(
    FICHIER_BPE,
    "r",
    encoding="utf-8-sig",
) as flux:
    premiere_ligne = flux.readline()

separateurs_valides = []

for separateur in [";", ",", "\t", "|"]:
    colonnes = next(
        csv.reader([premiere_ligne], delimiter=separateur)
    )

    colonnes = {
        normaliser_nom_colonne(colonne)
        for colonne in colonnes
    }

    if {"AN", "DEPCOM", "TYPEQU"}.issubset(colonnes):
        separateurs_valides.append(separateur)

if len(separateurs_valides) != 1:
    raise ValueError(
        "Impossible d'identifier un en-tête BPE détaillé. "
        "Colonnes attendues : AN, DEPCOM, TYPEQU. "
        f"Début du fichier : {premiere_ligne[:300]!r}"
    )

SEPARATEUR_BPE = separateurs_valides[0]

entete = pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    encoding="utf-8-sig",
    nrows=0,
)

noms_normalises = [
    normaliser_nom_colonne(colonne)
    for colonne in entete.columns
]

if len(noms_normalises) != len(set(noms_normalises)):
    raise ValueError("Colonnes BPE ambiguës après normalisation.")

print("Séparateur :", repr(SEPARATEUR_BPE))
print("Nombre de colonnes :", len(entete.columns))
print(noms_normalises)

Séparateur : ';'
Nombre de colonnes : 95
['AN', 'APET', 'NOMRS', 'CNOMRS', 'NUMVOIE', 'INDREP', 'TYPVOIE', 'LIBVOIE', 'CADR', 'CODPOS', 'DEPCOM', 'DEP', 'REG', 'LIBCOM', 'DOM', 'SDOM', 'TYPEQU', 'SIRET', 'STATUT_DIFFUSION', 'CANTINE', 'INTERNAT', 'RPI', 'EP', 'CL_PGE', 'SECT', 'SECTEUR', 'ACCES_AIRE_PRATIQUE', 'ACCES_LIBRE', 'ACCES_SANITAIRE', 'ACCES_VESTIAIRE', 'CAPACITE_D_ACCUEIL', 'PRES_DOUCHE', 'PRES_SANITAIRE', 'SAISONNIER', 'COUVERT', 'ECLAIRE', 'CATEGORIE', 'MULTIPLEXE', 'STRUCTURE_EXERCICE', 'SPECIALITE', 'ACCUEIL', 'ITINERANCE', 'MODE_GESTION', 'SSTYPHEB', 'TYPE', 'TYPERESTO', 'IMPLANTATION_STATION', 'GPL', 'CAPACITE', 'INDIC_CAPA', 'NBEQUIDENT', 'INDIC_NBEQUIDENT', 'NBSALLES', 'INDIC_NBSALLES', 'NBLIEUX', 'INDIC_NBLIEUX', 'NB_PDC', 'INDIC_NB_PDC', 'NB_PDC_PA', 'INDIC_NB_PDC_PA', 'NB_PDC_ACCELEREE', 'INDIC_NB_PDC_ACCELEREE', 'NB_PDC_LENTE', 'INDIC_NB_PDC_LENTE', 'NB_PDC_RAPIDE', 'INDIC_NB_PDC_RAPIDE', 'NB_PDC_ULTRARAPIDE', 'INDIC_NB_PDC_ULTRARAPIDE', 'NB_JOURS_OUVERT', 'INDIC_

In [7]:
#Lire par blocs

blocs_retenus = []
nombre_lignes_lues = 0

with pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    dtype="string",
    encoding="utf-8-sig",
    keep_default_na=False,
    chunksize=20_000,
) as lecteur:

    for numero_bloc, bloc in enumerate(lecteur, start=1):
        bloc.columns = [
            normaliser_nom_colonne(colonne)
            for colonne in bloc.columns
        ]

        bloc["RANG_SOURCE_J10B"] = np.arange(
            nombre_lignes_lues + 1,
            nombre_lignes_lues + len(bloc) + 1,
        )

        nombre_lignes_lues += len(bloc)

        types = nettoyer_texte(bloc["TYPEQU"]).str.upper()
        communes = normaliser_code_commune(bloc["DEPCOM"])

        masque = (
            types.isin(CATEGORIES)
            & communes.str[:2].isin(DEPARTEMENTS_IDF)
        )

        selection = bloc.loc[masque].copy()

        if not selection.empty:
            selection["TYPEQU_ANALYSE"] = types.loc[masque]
            selection["CODGEO_SOURCE"] = communes.loc[masque]
            blocs_retenus.append(selection)

        if numero_bloc % 25 == 0:
            print(f"{nombre_lignes_lues:,} lignes examinées")

if not blocs_retenus:
    raise ValueError("Aucun commerce du périmètre trouvé en IDF.")

commerces = pd.concat(
    blocs_retenus,
    ignore_index=True,
)

del blocs_retenus

annees = nettoyer_texte(commerces["AN"])

if not annees.eq("2025").fillna(False).all():
    raise ValueError(
        f"Millésimes inattendus : {annees.unique().tolist()}"
    )

print("Lignes source examinées :", nombre_lignes_lues)
print("Observations retenues :", len(commerces))

500,000 lignes examinées
1,000,000 lignes examinées
1,500,000 lignes examinées
2,000,000 lignes examinées
2,500,000 lignes examinées
Lignes source examinées : 2921770
Observations retenues : 27373


In [8]:
#Observer la tracabilité

HASH_BPE = sha256_fichier(FICHIER_BPE)

DATE_TRAITEMENT = datetime.now(timezone.utc).isoformat()

commerces["ID_OBSERVATION"] = (
    "BPE2025_"
    + HASH_BPE[:16]
    + "_"
    + commerces["RANG_SOURCE_J10B"].astype("string")
)

commerces["CATEGORIE_ANALYSE"] = (
    commerces["TYPEQU_ANALYSE"].map(CATEGORIES)
)

commerces["PERIMETRE_ANALYSE"] = np.where(
    commerces["TYPEQU_ANALYSE"].isin(CODES_ALIMENTAIRES_B2),
    "ALIMENTAIRE_B2",
    "GRANDES_SURFACES_ET_MIXTES",
)

commerces["SOURCE_DONNEES"] = "INSEE_BPE_2025"
commerces["STATUT_OUVERTURE_ACTUELLE"] = "NON_VERIFIE"

assert commerces["ID_OBSERVATION"].is_unique

display(
    commerces[
        [
            "ID_OBSERVATION",
            "TYPEQU_ANALYSE",
            "CATEGORIE_ANALYSE",
            "PERIMETRE_ANALYSE",
        ]
    ].head()
)

,ID_OBSERVATION,TYPEQU_ANALYSE,CATEGORIE_ANALYSE,PERIMETRE_ANALYSE
0,BPE2025_47d4f3a1fa8e37c5_94707,B104,HYPER_GRANDS_MAGASINS,GRANDES_SURFACES_ET_MIXTES
1,BPE2025_47d4f3a1fa8e37c5_94708,B104,HYPER_GRANDS_MAGASINS,GRANDES_SURFACES_ET_MIXTES
2,BPE2025_47d4f3a1fa8e37c5_94709,B105,SUPER_MULTI_COMMERCES,GRANDES_SURFACES_ET_MIXTES
3,BPE2025_47d4f3a1fa8e37c5_94710,B105,SUPER_MULTI_COMMERCES,GRANDES_SURFACES_ET_MIXTES
4,BPE2025_47d4f3a1fa8e37c5_94711,B105,SUPER_MULTI_COMMERCES,GRANDES_SURFACES_ET_MIXTES


In [9]:
#Normaliser les codes communaux

commerces["CODGEO"] = commerces["CODGEO_SOURCE"]

codes_arrondissements_paris = {
    f"751{numero:02d}" for numero in range(1, 21)
}

if codes_profil & codes_arrondissements_paris:
    raise ValueError(
        "Le profil contient des arrondissements parisiens. "
        "Cette version du notebook attend un profil communal."
    )

if "75056" in codes_profil:
    masque_paris = commerces["CODGEO"].isin(
        codes_arrondissements_paris
    )
    commerces.loc[masque_paris, "CODGEO"] = "75056"

if "93066" in codes_profil and "93059" not in codes_profil:
    commerces.loc[
        commerces["CODGEO"].eq("93059"),
        "CODGEO",
    ] = "93066"

commerces["COMMUNE_DANS_PROFIL"] = (
    commerces["CODGEO"].isin(codes_profil)
)

hors_profil = commerces.loc[
    ~commerces["COMMUNE_DANS_PROFIL"]
].copy()

sauvegarder_csv(
    hors_profil,
    DOSSIER_INTERIM / "commerces_hors_profil.csv",
)

print("Observations hors profil :", len(hors_profil))

if not hors_profil.empty:
    display(
        hors_profil[
            ["CODGEO_SOURCE", "CODGEO", "TYPEQU_ANALYSE"]
        ].drop_duplicates()
    )
    raise ValueError(
        "Des communes ne sont pas appariées. "
        "Examiner commerces_hors_profil.csv avant de poursuivre."
    )

Observations hors profil : 0


In [10]:
#Préparer les noms et adresses

colonnes_facultatives = [
    "SIRET", "NOMRS", "CNOMRS",
    "NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR",
    "CODPOS", "LATITUDE", "LONGITUDE",
]

for colonne in colonnes_facultatives:
    if colonne not in commerces.columns:
        commerces[colonne] = pd.Series(
            pd.NA,
            index=commerces.index,
            dtype="string",
        )

commerces["NOM_AFFICHAGE"] = (
    nettoyer_texte(commerces["NOMRS"])
    .fillna(nettoyer_texte(commerces["CNOMRS"]))
)

parties_adresse = commerces[
    ["NUMVOIE", "INDREP", "TYPVOIE", "LIBVOIE", "CADR"]
].apply(nettoyer_texte)

commerces["ADRESSE_RECOMPOSEE"] = (
    parties_adresse.fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)

display(
    commerces[
        ["NOM_AFFICHAGE", "ADRESSE_RECOMPOSEE", "CODGEO"]
    ].head(10)
)

,NOM_AFFICHAGE,ADRESSE_RECOMPOSEE,CODGEO
0,"MONOPRIX EXPLOITATION, PAR ABREVIATION MPX",21 AV DE L OPERA,75056
1,SAMARITAINE SAS,21 RUE DE LA MONNAIE,75056
2,SUPERMARCHE SAINT HONORE,20 PL DU MARCHE SAINT HONORE,75056
3,MINI LP 1,2 RUE DE MARENGO 2 6,75056
4,METATRON313,7 RUE MONTMARTRE,75056
5,"MONOPRIX EXPLOITATION, PAR ABREVIATION MPX",101 PTE BERGER FORUM DES HALLES,75056
6,ARS SEVEN,15 RUE DES HALLES,75056
7,MSH PROXI,77 RUE RAMBUTEAU,75056
8,MLK BUSINESS,165 RUE SAINT HONORE,75056
9,ALZERG,43 RUE DE RICHELIEU,75056


In [11]:
#Contrômer les SIRET repétés

commerces["SIRET_NETTOYE"] = nettoyer_texte(
    commerces["SIRET"]
)

commerces["SIRET_FORMAT_CONFORME"] = (
    commerces["SIRET_NETTOYE"]
    .str.fullmatch(r"\d{14}", na=False)
    & commerces["SIRET_NETTOYE"].ne("0" * 14).fillna(False)
)

commerces["SIRET_TYPE_REPETE"] = (
    commerces["SIRET_FORMAT_CONFORME"]
    & commerces.duplicated(
        subset=["SIRET_NETTOYE", "TYPEQU_ANALYSE"],
        keep=False,
    )
)

repetitions = commerces.loc[
    commerces["SIRET_TYPE_REPETE"]
].copy()

sauvegarder_csv(
    repetitions,
    DOSSIER_INTERIM / "siret_types_repetes.csv",
)

print(
    "SIRET au format conforme :",
    int(commerces["SIRET_FORMAT_CONFORME"].sum()),
)

print(
    "Observations avec SIRET et type répétés :",
    len(repetitions),
)

SIRET au format conforme : 27373
Observations avec SIRET et type répétés : 0


In [12]:
#Controler les coordonées

commerces["LATITUDE_NUM"] = convertir_nombre(
    commerces["LATITUDE"]
)

commerces["LONGITUDE_NUM"] = convertir_nombre(
    commerces["LONGITUDE"]
)

coordonnees_presentes = (
    commerces["LATITUDE_NUM"].notna()
    & commerces["LONGITUDE_NUM"].notna()
)

coordonnees_plausibles = (
    commerces["LATITUDE_NUM"].between(48.0, 49.3)
    & commerces["LONGITUDE_NUM"].between(1.4, 3.7)
).fillna(False)

commerces["STATUT_COORDONNEES"] = "ABSENTES_OU_NON_NUMERIQUES"

commerces.loc[
    coordonnees_presentes,
    "STATUT_COORDONNEES",
] = "HORS_RECTANGLE_CONTROLE"

commerces.loc[
    coordonnees_plausibles,
    "STATUT_COORDONNEES",
] = "PLAUSIBLES_A_VERIFIER"

controle_coordonnees = (
    commerces.groupby(
        ["CATEGORIE_ANALYSE", "STATUT_COORDONNEES"]
    )
    .size()
    .reset_index(name="NB_OBSERVATIONS")
)

sauvegarder_csv(
    controle_coordonnees,
    DOSSIER_INTERIM / "controle_coordonnees.csv",
)

display(controle_coordonnees)

,CATEGORIE_ANALYSE,STATUT_COORDONNEES,NB_OBSERVATIONS
0,AUTRES_ALIMENTAIRES,ABSENTES_OU_NON_NUMERIQUES,35
1,AUTRES_ALIMENTAIRES,PLAUSIBLES_A_VERIFIER,2385
2,BOISSONS,ABSENTES_OU_NON_NUMERIQUES,34
3,BOISSONS,PLAUSIBLES_A_VERIFIER,1287
4,BOUCHERIES,ABSENTES_OU_NON_NUMERIQUES,19
5,BOUCHERIES,PLAUSIBLES_A_VERIFIER,2633
6,BOULANGERIES_PATISSERIES,ABSENTES_OU_NON_NUMERIQUES,318
7,BOULANGERIES_PATISSERIES,PLAUSIBLES_A_VERIFIER,8047
8,EPICERIES,ABSENTES_OU_NON_NUMERIQUES,63
9,EPICERIES,PLAUSIBLES_A_VERIFIER,6906


In [13]:
#Compter les observations par commune

comptages = pd.crosstab(
    commerces["CODGEO"],
    commerces["TYPEQU_ANALYSE"],
)

comptages = comptages.reindex(
    columns=list(CATEGORIES),
    fill_value=0,
)

comptages = (
    comptages.rename(columns=COLONNES_COMPTAGE)
    .reset_index()
)

indicateurs = profil[["CODGEO"]].merge(
    comptages,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

colonnes_categories = list(COLONNES_COMPTAGE.values())

indicateurs[colonnes_categories] = (
    indicateurs[colonnes_categories]
    .fillna(0)
    .astype("int64")
)

display(indicateurs.head())

,CODGEO,NB_OBS_HYPER_GRANDS_MAGASINS_BPE2025,NB_OBS_SUPER_MULTI_COMMERCES_BPE2025,NB_OBS_SUPERETTES_BPE2025,NB_OBS_EPICERIES_BPE2025,NB_OBS_BOUCHERIES_BPE2025,NB_OBS_SURGELES_BPE2025,NB_OBS_POISSONNERIES_BPE2025,NB_OBS_BOULANGERIES_PATISSERIES_BPE2025,NB_OBS_FRUITS_LEGUMES_BPE2025,NB_OBS_BOISSONS_BPE2025,NB_OBS_AUTRES_ALIMENTAIRES_BPE2025
0,75056,44,715,368,2106,649,125,102,2307,387,715,1065
1,77001,0,0,0,0,0,0,0,0,0,0,2
2,77002,0,0,0,0,0,0,0,0,0,0,0
3,77003,0,0,0,0,0,0,0,0,0,0,0
4,77004,0,0,0,0,0,0,0,0,0,0,0


In [14]:
#Construire les regroupements utiles

colonnes_b2 = [
    COLONNES_COMPTAGE[code]
    for code in CODES_ALIMENTAIRES_B2
]

colonnes_grandes_surfaces = [
    COLONNES_COMPTAGE[code]
    for code in CODES_GRANDES_SURFACES
]

indicateurs["NB_OBS_ALIMENTAIRES_B2_BPE2025"] = (
    indicateurs[colonnes_b2].sum(axis=1)
)

indicateurs["NB_OBS_EPICERIES_SUPERETTES_BPE2025"] = (
    indicateurs[COLONNES_COMPTAGE["B201"]]
    + indicateurs[COLONNES_COMPTAGE["B202"]]
)

indicateurs["NB_OBS_GRANDES_SURFACES_MIXTES_BPE2025"] = (
    indicateurs[colonnes_grandes_surfaces].sum(axis=1)
)

indicateurs["NB_OBS_PERIMETRE_J10B_BPE2025"] = (
    indicateurs[colonnes_categories].sum(axis=1)
)

nombre_categories_b2 = (
    indicateurs[colonnes_b2].gt(0).sum(axis=1)
)

indicateurs["NB_CATEGORIES_ALIMENTAIRES_B2_PRESENTES"] = (
    nombre_categories_b2
)

assert (
    indicateurs["NB_OBS_PERIMETRE_J10B_BPE2025"]
    ==
    indicateurs["NB_OBS_ALIMENTAIRES_B2_BPE2025"]
    + indicateurs["NB_OBS_GRANDES_SURFACES_MIXTES_BPE2025"]
).all()

assert (
    indicateurs["NB_OBS_PERIMETRE_J10B_BPE2025"].sum()
    == len(commerces)
)

print("Agrégation cohérente.")

Agrégation cohérente.


In [15]:
#Joindre au profil J10 

nouvelles_colonnes = set(indicateurs.columns) - {"CODGEO"}
collisions = nouvelles_colonnes & set(profil.columns)

if collisions:
    raise ValueError(
        "Ces colonnes existent déjà dans le profil d'entrée : "
        + ", ".join(sorted(collisions))
    )

profil_j10b = profil.merge(
    indicateurs,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

assert len(profil_j10b) == len(profil)
assert profil_j10b["CODGEO"].is_unique

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10b[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

print("Jointure réussie :", len(profil_j10b), "communes.")

Jointure réussie : 1266 communes.


In [17]:
# À renseigner si plusieurs colonnes sont proposées.
# Exemple : COLONNE_POPULATION = "P23_POP"
COLONNE_POPULATION = None

candidats_population = [
    colonne
    for colonne in [
        "POPULATION_2022"
    ]
    if colonne in profil_j10b.columns
]

if COLONNE_POPULATION is None:
    if len(candidats_population) == 1:
        COLONNE_POPULATION = candidats_population[0]
    else:
        print("Candidats :", candidats_population)
        print("Colonnes disponibles :", profil_j10b.columns.tolist())

        raise ValueError(
            "Renseigne COLONNE_POPULATION avec la population "
            "de référence utilisée dans ton profil."
        )

if COLONNE_POPULATION not in profil_j10b.columns:
    raise ValueError("La colonne de population choisie n'existe pas.")

population = convertir_nombre(
    profil_j10b[COLONNE_POPULATION]
)

population_valide = population.gt(0).fillna(False)
denominateur = population.where(population_valide)

print("Population utilisée :", COLONNE_POPULATION)
print(
    "Communes sans population positive exploitable :",
    int((~population_valide).sum()),
)

Population utilisée : POPULATION_2022
Communes sans population positive exploitable : 0


In [18]:
# Total des observations du périmètre J10b.
# Si une commune n'en possède aucune, ses parts restent manquantes.
total_commerces = profil_j10b[
    "NB_OBS_PERIMETRE_J10B_BPE2025"
].replace(0, np.nan)

colonnes_parts_types = []

for code, categorie in CATEGORIES.items():
    colonne_nombre = COLONNES_COMPTAGE[code]

    colonne_part = (
        f"PART_{categorie}_PERIMETRE_J10B_PCT"
    )

    profil_j10b[colonne_part] = (
        profil_j10b[colonne_nombre]
        .div(total_commerces)
        .mul(100)
    )

    colonnes_parts_types.append(colonne_part)

print("Parts calculées :", len(colonnes_parts_types))

Parts calculées : 11


In [21]:
#Calculer les densités

MESURES_DENSITE = {
    "NB_OBS_ALIMENTAIRES_B2_BPE2025":
        "DENSITE_OBS_ALIMENTAIRES_B2_10000_HAB",

    "NB_OBS_EPICERIES_SUPERETTES_BPE2025":
        "DENSITE_OBS_EPICERIES_SUPERETTES_10000_HAB",

    COLONNES_COMPTAGE["B207"]:
        "DENSITE_OBS_BOULANGERIES_10000_HAB",

    "NB_OBS_GRANDES_SURFACES_MIXTES_BPE2025":
        "DENSITE_OBS_GRANDES_SURFACES_MIXTES_10000_HAB",
}

for colonne_nombre, colonne_densite in MESURES_DENSITE.items():
    if colonne_densite in profil_j10b.columns:
        raise ValueError(f"Colonne déjà présente : {colonne_densite}")

    profil_j10b[colonne_densite] = (
        profil_j10b[colonne_nombre]
        .div(denominateur)
        .mul(10_000)
    )

In [22]:
#Examiner les résultats

colonne_nom = next(
    (
        colonne
        for colonne in ["LIBELLE", "NOM_COMMUNE", "LIBGEO", "NOM_COM"]
        if colonne in profil_j10b.columns
    ),
    None,
)

colonnes_affichage = ["CODGEO"]

if colonne_nom:
    colonnes_affichage.append(colonne_nom)

colonnes_affichage += [
    COLONNE_POPULATION,
    "NB_OBS_ALIMENTAIRES_B2_BPE2025",
    "NB_OBS_EPICERIES_SUPERETTES_BPE2025",
    COLONNES_COMPTAGE["B207"],
    "NB_OBS_GRANDES_SURFACES_MIXTES_BPE2025",
    "DENSITE_OBS_ALIMENTAIRES_B2_10000_HAB",
]

print("Plus grands nombres d'observations alimentaires B2 :")

display(
    profil_j10b.sort_values(
        "NB_OBS_ALIMENTAIRES_B2_BPE2025",
        ascending=False,
    )[colonnes_affichage].head(20)
)

print("Densités — communes d'au moins 5 000 habitants :")

display(
    profil_j10b.loc[population.ge(5_000).fillna(False)]
    .sort_values(
        "DENSITE_OBS_ALIMENTAIRES_B2_10000_HAB",
        ascending=False,
    )[colonnes_affichage]
    .head(20)
)

Plus grands nombres d'observations alimentaires B2 :


,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_ALIMENTAIRES_B2_BPE2025,NB_OBS_EPICERIES_SUPERETTES_BPE2025,NB_OBS_BOULANGERIES_PATISSERIES_BPE2025,NB_OBS_GRANDES_SURFACES_MIXTES_BPE2025,DENSITE_OBS_ALIMENTAIRES_B2_10000_HAB
0,75056,Paris,2113705.0,7824,2474,2307,759,37.015572
1027,93066,Saint-Denis,148907.0,368,177,115,17,24.713412
997,93001,Aubervilliers,89489.0,253,140,58,12,28.271631
1016,93048,Montreuil,110758.0,251,102,82,21,22.66202
965,92012,Boulogne-Billancourt,120205.0,239,75,72,27,19.8827
1088,95018,Argenteuil,107135.0,205,77,78,13,19.134737
754,78646,Versailles,83918.0,190,53,63,17,22.64115
962,92004,Asnières-sur-Seine,91457.0,181,62,68,22,19.790721
1070,94068,Saint-Maur-des-Fossés,76010.0,178,50,64,16,23.417971
1006,93027,La Courneuve,47086.0,163,85,49,7,34.617508


Densités — communes d'au moins 5 000 habitants :


,CODGEO,NOM_COMMUNE,POPULATION_2022,NB_OBS_ALIMENTAIRES_B2_BPE2025,NB_OBS_EPICERIES_SUPERETTES_BPE2025,NB_OBS_BOULANGERIES_PATISSERIES_BPE2025,NB_OBS_GRANDES_SURFACES_MIXTES_BPE2025,DENSITE_OBS_ALIMENTAIRES_B2_10000_HAB
0,75056,Paris,2113705.0,7824,2474,2307,759,37.015572
1141,95210,Enghien-les-Bains,11594.0,42,11,14,7,36.225634
176,77186,Fontainebleau,15787.0,56,11,19,5,35.472224
1068,94065,Rungis,5669.0,20,5,7,1,35.279591
1006,93027,La Courneuve,47086.0,163,85,49,7,34.617508
662,78401,Meulan-en-Yvelines,8996.0,30,12,4,2,33.348155
1089,95019,Arnouville,14898.0,48,21,18,3,32.21909
125,77131,Coulommiers,15696.0,50,18,16,4,31.85525
928,91587,Saulx-les-Chartreux,6610.99999999999,21,5,9,5,31.76524
173,77183,La Ferté-sous-Jouarre,10004.0,31,13,8,3,30.987605


In [23]:
#Préparer la qualification commerciale locale

FICHIER_QUALIFICATION = (
    DOSSIER_INTERIM / "qualification_commerciale_locale.csv"
)

colonnes_qualification = [
    "ID_OBSERVATION",
    "SIRET",
    "NOM_VERIFIE",
    "OFFRE_DEJEUNER",
    "VENTE_A_EMPORTER",
    "CONSOMMATION_SUR_PLACE",
    "PRIX_FORMULE_DEJEUNER_EUR",
    "DATE_VERIFICATION",
    "SOURCE_URL",
    "COMMENTAIRE",
]

if not FICHIER_QUALIFICATION.exists():
    sauvegarder_csv(
        pd.DataFrame(columns=colonnes_qualification),
        FICHIER_QUALIFICATION,
    )

print("Fichier de qualification :", FICHIER_QUALIFICATION)

Fichier de qualification : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j10b\qualification_commerciale_locale.csv


In [24]:
#Contrôles finaux

colonnes_effectifs = [
    colonne
    for colonne in indicateurs.columns
    if colonne != "CODGEO"
]

assert len(profil_j10b) == len(profil)
assert profil_j10b["CODGEO"].is_unique
assert set(profil_j10b["CODGEO"]) == codes_profil

assert commerces["ID_OBSERVATION"].is_unique
assert commerces["COMMUNE_DANS_PROFIL"].all()

assert profil_j10b[colonnes_effectifs].notna().all().all()
assert profil_j10b[colonnes_effectifs].ge(0).all().all()

assert (
    profil_j10b["NB_OBS_PERIMETRE_J10B_BPE2025"].sum()
    == len(commerces)
)

for colonne in MESURES_DENSITE.values():
    valeurs = profil_j10b[colonne].dropna()

    assert np.isfinite(valeurs.astype(float)).all()
    assert valeurs.ge(0).all()

    assert profil_j10b.loc[
        ~population_valide, colonne
    ].isna().all()

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j10b[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

controle_jointures = pd.DataFrame([{
    "COMMUNES_ENTREE": len(profil),
    "COMMUNES_SORTIE": len(profil_j10b),
    "OBSERVATIONS_DETAIL": len(commerces),
    "OBSERVATIONS_HORS_PROFIL": len(hors_profil),
    "OBSERVATIONS_SIRET_TYPE_REPETE": len(repetitions),
    "ETAT": (
        "CONTROLES_STRUCTURELS_OK"
        if repetitions.empty
        else "CONTROLES_STRUCTURELS_OK_REPETITIONS_A_EXAMINER"
    ),
}])

sauvegarder_csv(
    controle_jointures,
    DOSSIER_INTERIM / "controle_jointures.csv",
)

display(controle_jointures)

,COMMUNES_ENTREE,COMMUNES_SORTIE,OBSERVATIONS_DETAIL,OBSERVATIONS_HORS_PROFIL,OBSERVATIONS_SIRET_TYPE_REPETE,ETAT
0,1266,1266,27373,0,0,CONTROLES_STRUCTURELS_OK


In [25]:
#Enregistrer les données et métadonnées

sauvegarder_csv(commerces, FICHIER_DETAIL)
sauvegarder_csv(indicateurs, FICHIER_COMMUNES)
sauvegarder_csv(profil_j10b, FICHIER_SORTIE)

metadonnees = {
    "notebook": "10b_commerces_alimentaires_idf.ipynb",
    "date_traitement_utc": DATE_TRAITEMENT,
    "source": "INSEE - BPE 2025",
    "url_nomenclature": (
        "https://www.insee.fr/fr/metadonnees/source/fichier/"
        "BPE25_liste_hierarchisee_TYPEQU.html"
    ),
    "fichier_bpe": str(FICHIER_BPE),
    "sha256_bpe": HASH_BPE,
    "profil_entree": str(FICHIER_PROFIL),
    "sha256_profil_entree": sha256_fichier(FICHIER_PROFIL),
    "profil_sortie": str(FICHIER_SORTIE),
    "millesime_bpe": 2025,
    "categories": CATEGORIES,
    "colonne_population": COLONNE_POPULATION,
    "nombre_observations": len(commerces),
    "nombre_communes": len(profil_j10b),
    "observations_siret_type_repete": len(repetitions),
    "observations_hors_profil": len(hors_profil),
    "deduplication_automatique": False,
    "limites": [
        "Comptages d'observations BPE.",
        "Ouverture actuelle non vérifiée.",
        "B104 et B105 conservés séparément du groupe alimentaire B2.",
        "Coordonnées seulement contrôlées par rectangle géographique.",
        "Offre de déjeuner non déduite automatiquement du type BPE.",
        "Densités calculées sur la population de référence du profil.",
    ],
}

with open(
    DOSSIER_INTERIM / "metadata_j10b.json",
    "w",
    encoding="utf-8",
) as flux:
    json.dump(
        metadonnees,
        flux,
        ensure_ascii=False,
        indent=2,
    )

print("Détail :", FICHIER_DETAIL)
print("Agrégation :", FICHIER_COMMUNES)
print("Profil enrichi :", FICHIER_SORTIE)
print("J10b enregistré.")

Détail : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\commerces_alimentaires_idf_j10b.csv
Agrégation : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j10b\commerces_alimentaires_communes.csv
Profil enrichi : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10b.csv
J10b enregistré.
